In [1]:
import warnings

# Mute all warnings globally
warnings.filterwarnings("ignore")

# National Pipeline — `NationalScenarioGenerator`

The standard national simulation combining **Forest Management (FM)**, **Afforestation (AF)**, and **Scenarios (SC)** using the bundled Irish database, running to 2070.

This notebook mirrors `tests/examples/nsg_example.py`. Output is **archive-only**: call `export_archive()` to write a self-describing SQLite `.db`.

## Inputs

Provide a scenario DataFrame and an afforestation DataFrame. The example ships sample CSVs under `tests/examples/data/input/`.

In [2]:
import pandas as pd
from goblin_cbm_runner.scenario_generator import NationalScenarioGenerator

scenario_df = pd.read_csv('../../tests/examples/data/input/scenario_dataframe.csv')
afforest_df = pd.read_csv('../../tests/examples/data/input/cbm_afforestation.csv', index_col=0)

## Run the simulation

`comprehensive=False` runs the lightweight flux pipeline. Set `comprehensive=True` to also capture the pools / flux / state validation tables.

In [3]:
nsg = NationalScenarioGenerator(
    scenario_data=scenario_df,
    afforestation_data=afforest_df,
    config={'afforest_delay': 5, 'annual_rate_pre_delay': 1200},
    comprehensive=False,
)

results = nsg.run_flux_simulation()
results.head()

INFO:goblin_cbm_runner.runners.fm_runner:Generating FM input data...
INFO:goblin_cbm_runner.runners.fm_runner:FM input data will be saved to: /home/colm/Dropbox/projects/FORESIGHT/packages/cbm_runner/src/goblin_cbm_runner/data/FM_input
INFO:goblin_cbm_runner.runners.fm_runner:FM input data generated


🚀 Starting FM Simulation...


INFO:goblin_cbm_runner.runners.af_runner:Generating AF input data...
INFO:goblin_cbm_runner.runners.af_runner:AF input data will be saved to: /home/colm/Dropbox/projects/FORESIGHT/packages/cbm_runner/src/goblin_cbm_runner/data/AF_input
INFO:goblin_cbm_runner.runners.af_runner:AF input data generated


FM Simulation Complete.
Starting AF Simulation...


INFO:goblin_cbm_runner.runners.sc_runner:Generating SC input data for scenario 0...
INFO:goblin_cbm_runner.runners.sc_runner:SC input data will be saved to: /home/colm/Dropbox/projects/FORESIGHT/packages/cbm_runner/src/goblin_cbm_runner/data/SC_input


...

INFO:goblin_cbm_runner.runners.sc_runner:Generating SC input data for scenario 1...
INFO:goblin_cbm_runner.runners.sc_runner:SC input data will be saved to: /home/colm/Dropbox/projects/FORESIGHT/packages/cbm_runner/src/goblin_cbm_runner/data/SC_input


Scenario 0 Simulation Complete.


INFO:goblin_cbm_runner.runners.sc_runner:SC input data generated for scenario 1


Starting Scenario 1 Simulation...
Scenario 1 Simulation Complete.


,Year,AGB,BGB,Deadwood,Litter,Soil,Harvest,Total Ecosystem,Scenario
0,1991,7263.311101,1759.130211,8.504021,343.097664,-1734.363021,0.0,7639.679976,-1
1,1992,22253.850184,5250.505060,52.796405,1556.895918,-5776.696748,0.0,23337.350820,-1
2,1993,44303.328764,10481.865799,175.098926,3892.237344,-8829.050404,0.0,50023.480430,-1
3,1994,69440.947422,17314.024378,424.060031,7326.565980,-11232.248424,0.0,83273.349387,-1
4,1995,102710.631872,25607.246189,844.222681,11850.327134,-14254.706643,0.0,126757.721232,-1


## Save the archive

All inputs, results, harvest/NAI summaries, and (in comprehensive mode) validation tables are written to one SQLite file.

In [4]:
nsg.export_archive('./nsg_simulation_archive.db')

Each row in the archive carries a `runner_type` (`FM`, `AF`, `SC`, `combined_af_fm`, `combined_sc_af_fm`) so downstream queries can select the baseline or a given scenario.